# 03. Modelado clásico y comparación de algoritmos

**MAP - Charting Student Math Misunderstandings**

En este cuaderno se construyen y comparan seis modelos de clasificación. El objetivo es
determinar cuáles resuelven mejor el problema y, sobre todo, cuánto de su rendimiento proviene
de leer realmente la explicación del estudiante.

El trabajo se organiza así:

1. **Preparación.** Carga del conjunto limpio y definición de las dos etapas del problema.
2. **La métrica.** Qué es MAP@3 y por qué la competencia la usa.
3. **Ajuste de hiperparámetros.** Búsqueda acotada para cada modelo.
4. **Etapa 1.** Predicción de `Category`, con los dos esquemas de validación.
5. **Etapa 2.** Predicción de `Misconception`.
6. **Objetivo compuesto.** Composición de ambas etapas y evaluación con MAP@3.
7. **Comparación y conclusiones.**

Los modelos y las métricas se guardan para que los usen el cuaderno de comparación y la
aplicación.

## 0. Arranque

Este cuaderno funciona igual en **Google Colab** y en una instalación local.

En Colab la celda siguiente clona el repositorio, instala las dependencias y deja el proyecto
listo. Para que la descarga de los datos funcione hay que registrar antes el token de Kaggle
como secreto: en el panel izquierdo de Colab, icono de la llave, agregar un secreto llamado
`KAGGLE_ACCESS_TOKEN` con el contenido de `~/.kaggle/access_token` y habilitar el acceso desde
el cuaderno.

En local la celda no hace nada salvo ubicar la raíz del proyecto.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPOSITORIO = "https://github.com/Tunchxz/math-misconception-map.git"
CARPETA = "math-misconception-map"

if EN_COLAB:
    if not Path(CARPETA).exists():
        subprocess.run(["git", "clone", "--depth", "1", "-q", REPOSITORIO, CARPETA], check=True)
    os.chdir(f"/content/{CARPETA}")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True
    )
    try:
        from google.colab import userdata

        os.environ["KAGGLE_ACCESS_TOKEN"] = userdata.get("KAGGLE_ACCESS_TOKEN")
        print("Token de Kaggle cargado desde los secretos de Colab.")
    except Exception:
        print("Registre el secreto KAGGLE_ACCESS_TOKEN para poder descargar los datos.")

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

print("Entorno:", "Google Colab" if EN_COLAB else "local")
print("Raíz del proyecto:", RAIZ)

In [ ]:
import sys, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

import joblib

from src.config import MODELS, DATA_TRANSFORMED, SEMILLA, SIN_MISCONCEPTION
from src.data.make_dataset import asegurar_datos, guardar_transformado
from src.features import build_features as bf
from src.models import train_model as tm
from src.visualization.visualize import set_estilo

set_estilo()
pd.set_option("display.width", 170)
pd.set_option("display.max_colwidth", 60)

MODELS.mkdir(parents=True, exist_ok=True)
print("Semilla:", SEMILLA)

## 1. Preparación de los datos

Se parte del conjunto limpio producido en el cuaderno 01. Sobre él se aplica la unificación de
las dos grafías de la misma etiqueta que se detectó durante la limpieza.

In [ ]:
train = bf.preparar_objetivos(asegurar_datos())

print("Observaciones:", f"{len(train):,}")
print("Clases de Category:", train["Category"].nunique())
print("Etiquetas de misconception tras unificar:", train["misconception_unificada"].nunique() - 1)
print("Objetivo compuesto, combinaciones:", train["objetivo_unificado"].nunique())
print()
print("Correspondencia de las grafías unificadas:")
print(bf.tabla_alias(train).to_string(index=False))

In [ ]:
COLUMNAS = ["explicacion_min", "QuestionId", "MC_Answer"] + bf.NUMERICAS

X = train[COLUMNAS]
grupos = train["QuestionId"].values

y_etapa1 = train["Category"].values

con_error = bf.filas_con_misconception(train)
X2 = con_error[COLUMNAS]
y_etapa2 = con_error["misconception_unificada"].values
grupos2 = con_error["QuestionId"].values

print(f"Etapa 1: {X.shape[0]:,} filas, {len(set(y_etapa1))} clases")
print(f"Etapa 2: {X2.shape[0]:,} filas, {len(set(y_etapa2))} clases")
print(f"Preguntas distintas: {len(set(grupos))}")

### Por qué dos etapas

El análisis exploratorio mostró que las dos partes del problema tienen dificultad muy distinta.
Decidir la categoría general depende poco de la pregunta concreta, mientras que identificar el
error conceptual específico depende casi por completo de ella. Separarlas permite medir cada una
con su propia métrica en lugar de promediar dos cosas distintas.

La etapa 2 se entrena únicamente con las filas que efectivamente tienen un error conceptual,
porque en las demás la pregunta no aplica.

## 2. La métrica MAP@3

La competencia admite hasta tres predicciones por observación, separadas por espacio, y las
evalúa con la precisión promedio media hasta la tercera posición. Cada observación aporta 1 si
la respuesta correcta está en primer lugar, un medio si está en segundo, un tercio si está en
tercero y cero si no aparece.

Esto reconoce que la tarea es ambigua. Antes de usarla se verifica la implementación con un
ejemplo construido a mano.

In [ ]:
clases_demo = np.array(["A", "B", "C", "D"])
puntajes_demo = np.array([
    [0.1, 0.5, 0.3, 0.1],   # orden: B, C, A
    [0.6, 0.2, 0.1, 0.1],   # orden: A, B, C
    [0.1, 0.1, 0.1, 0.7],   # orden: D, A, B
])
verdad_demo = np.array(["B", "C", "X"])

print("Las tres candidatas de cada fila:", tm.formato_envio(puntajes_demo, clases_demo))
print()
print("Fila 1: 'B' en posición 1 aporta 1/1 = 1.000")
print("Fila 2: 'C' en posición 3 aporta 1/3 = 0.333")
print("Fila 3: 'X' no aparece, aporta 0.000")
print(f"Promedio esperado: {(1 + 1/3 + 0) / 3:.4f}")
print(f"Valor calculado:   {tm.map_at_k(verdad_demo, puntajes_demo, clases_demo, k=3):.4f}")

## 3. Los modelos comparados

Se comparan seis modelos. Los dos primeros son líneas base que no leen el texto y existen para
poder medir, por diferencia, cuánto aporta leerlo.

In [ ]:
catalogo = tm.catalogo_modelos()

resumen_modelos = pd.DataFrame([
    {
        "modelo": m.nombre,
        "usa_el_texto": "sí" if m.usa_texto else "no",
        "hiperparametros_a_ajustar": ", ".join(m.rejilla) if m.rejilla else "ninguno",
    }
    for m in catalogo
])
resumen_modelos

## 4. Ajuste de hiperparámetros de la etapa 1

Para cada modelo se prueba su rejilla sobre una única partición de retención. Se usa una sola
partición y no validación cruzada completa para acotar el tiempo de cómputo, que es la
restricción práctica de este proyecto.

In [ ]:
mejores_params = {}
tablas_tuning = {}

for m in catalogo:
    if not m.rejilla:
        mejores_params[m.nombre] = {}
        continue
    params, tabla = tm.tunear(m, X, y_etapa1, grupos)
    mejores_params[m.nombre] = params
    tablas_tuning[m.nombre] = tabla
    print(f"{m.nombre:24s} -> {params}")

In [ ]:
for nombre, tabla in tablas_tuning.items():
    print(f"\n=== {nombre} ===")
    print(tabla.round(4).to_string(index=False))

## 5. Ajuste de la etapa 2

Se repite la búsqueda sobre las filas que sí tienen un error conceptual.

In [ ]:
mejores_params_e2 = {}
for m in catalogo:
    if not m.rejilla:
        mejores_params_e2[m.nombre] = {}
        continue
    params, _ = tm.tunear(m, X2, y_etapa2, grupos2)
    mejores_params_e2[m.nombre] = params
    print(f"{m.nombre:24s} -> {params}")

## 6. Validación de los modelos

Cada modelo se valida con los dos esquemas. El esquema **aleatorio** reparte las filas al azar,
de modo que las mismas preguntas aparecen en entrenamiento y en prueba. El esquema **agrupado**
deja preguntas completas fuera del entrenamiento, que es la situación real de la competencia.

Las tres etapas se miden en un solo recorrido de particiones. Cada modelo se ajusta una vez por
partición y ese ajuste se reutiliza para las tres mediciones, lo que evita repetir el
entrenamiento tres veces.

In [ ]:
filas = []
for m in catalogo:
    for esquema in tm.ESQUEMAS:
        resultados = tm.evaluar_completo(
            m, train, COLUMNAS, esquema,
            mejores_params.get(m.nombre, {}), mejores_params_e2.get(m.nombre, {}),
        )
        filas.extend(resultados)
        linea = " | ".join(
            f"{r['etapa'][0]}: f1m={r['f1_macro']:.3f} map3={r['map@3']:.3f}"
            for r in resultados
        )
        print(f"{m.nombre:24s} {esquema:10s} {linea}  ({resultados[0]['segundos']:.0f}s)")

metricas = pd.DataFrame(filas)

### Etapa 1: predicción de `Category`

In [ ]:
def vista(etapa):
    "Tabla comparativa de una etapa, con los dos esquemas lado a lado."
    sub = metricas[metricas["etapa"] == etapa]
    return (
        sub.pivot_table(index="modelo", columns="esquema",
                        values=["f1_macro", "f1_ponderado", "map@3"])
        .reindex([m.nombre for m in catalogo])
        .round(4)
    )

vista("1. Category")

In [ ]:
def brecha(etapa):
    "Diferencia de MAP@3 entre la validacion aleatoria y la agrupada."
    sub = metricas[metricas["etapa"] == etapa]
    p = sub.pivot_table(index="modelo", columns="esquema", values="map@3")
    return (
        p.assign(
            brecha=lambda d: (d["aleatorio"] - d["agrupado"]).round(4),
            caida_pct=lambda d: ((d["aleatorio"] - d["agrupado"]) / d["aleatorio"] * 100).round(1),
        )
        .reindex([m.nombre for m in catalogo])
        .round(4)
    )

print("Caída de MAP@3 al pasar a preguntas nunca vistas, etapa 1:")
brecha("1. Category")

### Etapa 2: predicción de `Misconception`

Aquí el esquema agrupado es especialmente exigente. El análisis exploratorio mostró que la
mayoría de las etiquetas aparecen en una sola pregunta, de modo que al dejar esa pregunta fuera
del entrenamiento la etiqueta se vuelve imposible de predecir.

In [ ]:
vista("2. Misconception")

In [ ]:
print("Clases que el modelo nunca vio durante el entrenamiento, promedio por partición:")
print(
    metricas[metricas["etapa"] == "2. Misconception"]
    .pivot_table(index="modelo", columns="esquema", values="clases_no_vistas")
    .round(1).to_string()
)
print()
print(f"Total de etiquetas distintas en la etapa 2: {len(set(y_etapa2))}")

### Objetivo compuesto

La competencia evalúa la etiqueta completa `Category:Misconception`. Se compone a partir de las
dos etapas multiplicando sus probabilidades: la probabilidad de una combinación es la
probabilidad de la categoría por la probabilidad del error conceptual dado que lo hay. Para las
categorías que no admiten error conceptual, la combinación termina en `NA` y conserva la
probabilidad de la categoría.

In [ ]:
vista("3. Compuesto")

In [ ]:
print("Caída de MAP@3 en el objetivo compuesto:")
brecha("3. Compuesto")

## 7. Cuánto aporta leer la explicación

Esta es la comparación central del proyecto. Se contrasta el mejor modelo que lee el texto
contra la línea base que solo conoce la pregunta y la opción seleccionada.

In [ ]:
base = "Linea base por contexto"
nombres_con_texto = [m.nombre for m in catalogo if m.usa_texto]

comparacion = []
for etapa in sorted(metricas["etapa"].unique()):
    for esquema in tm.ESQUEMAS:
        sub = metricas[(metricas["etapa"] == etapa) & (metricas["esquema"] == esquema)]
        con_texto = sub[sub["modelo"].isin(nombres_con_texto)].dropna(subset=["map@3"])
        linea = sub[sub["modelo"] == base]
        if con_texto.empty or linea.empty:
            continue
        mejor = con_texto.loc[con_texto["map@3"].idxmax()]
        comparacion.append({
            "etapa": etapa,
            "esquema": esquema,
            "mejor_con_texto": mejor["modelo"],
            "map3_con_texto": round(mejor["map@3"], 4),
            "map3_solo_contexto": round(linea.iloc[0]["map@3"], 4),
            "ganancia": round(mejor["map@3"] - linea.iloc[0]["map@3"], 4),
        })

comparacion = pd.DataFrame(comparacion)
comparacion

## 8. Guardado de resultados y modelos

Se guardan las métricas de todos los modelos y los artefactos de cada uno, para que los usen el
cuaderno de comparación y la aplicación.

In [ ]:
guardar_transformado(metricas, "metricas_clasicos")
guardar_transformado(comparacion, "comparacion_texto_contexto")
guardar_transformado(bf.tabla_alias(train), "tabla_alias")

mejor_nombre = (
    metricas[(metricas["etapa"] == "3. Compuesto") & (metricas["esquema"] == "agrupado")]
    .sort_values("map@3", ascending=False).iloc[0]["modelo"]
)
print("Mejor modelo en el objetivo compuesto con validación agrupada:", mejor_nombre)

In [ ]:
sub_e2 = train[train["misconception_unificada"] != SIN_MISCONCEPTION]
guardados = []

for m in catalogo:
    if m.nombre == "Linea base (mayoritaria)":
        continue
    e1 = tm.entrenar_final(m, X, y_etapa1, mejores_params.get(m.nombre, {}))
    e2 = tm.entrenar_final(m, sub_e2[COLUMNAS], sub_e2["misconception_unificada"].values,
                           mejores_params_e2.get(m.nombre, {}))
    ruta = MODELS / f"{m.nombre.lower().replace(' ', '_')}.joblib"
    joblib.dump({"etapa1": e1, "etapa2": e2, "nombre": m.nombre}, ruta, compress=3)
    guardados.append({"modelo": m.nombre, "archivo": ruta.name,
                      "MB": round(ruta.stat().st_size / 1e6, 1)})

joblib.dump(
    {"columnas": COLUMNAS, "params_e1": mejores_params, "params_e2": mejores_params_e2},
    MODELS / "configuracion.joblib",
)
pd.DataFrame(guardados)

### Llevar los artefactos a la máquina local

Al ejecutar en Colab, los modelos y las métricas viven en un entorno temporal. La celda
siguiente los empaqueta y los descarga para que la aplicación pueda usarlos en local.

In [ ]:
import shutil

paquete = Path("artefactos_modelado")
paquete.mkdir(exist_ok=True)
shutil.copytree(MODELS, paquete / "models", dirs_exist_ok=True)
shutil.copytree(DATA_TRANSFORMED, paquete / "transformed", dirs_exist_ok=True)
shutil.make_archive("artefactos_modelado", "zip", paquete)

tamano = Path("artefactos_modelado.zip").stat().st_size / 1e6
print(f"artefactos_modelado.zip generado ({tamano:.1f} MB)")

if EN_COLAB:
    from google.colab import files

    files.download("artefactos_modelado.zip")
    print()
    print("En el repositorio local, descomprima así:")
    print("  unzip -o artefactos_modelado.zip -d /tmp/artefactos")
    print("  cp -r /tmp/artefactos/models/. models/")
    print("  cp -r /tmp/artefactos/transformed/. data/transformed/")